# llm-finetune-serve — Colab driver

This notebook stays thin on purpose: clone, install, call scripts. All logic
lives in `src/`. If you find yourself writing real code here, it belongs in the
repo instead.

## Two ways to run it

**In the browser:** open this notebook from GitHub, then Runtime → Change
runtime type → GPU.

**From VS Code** (no browser tab): install the official **Google Colab**
extension (publisher: Google), open this file locally, then kernel picker →
`Colab` → `Auto Connect`, and pick a GPU runtime.

Either way the kernel runs on a Colab VM, and the extension does **not** sync
local files to it — so your `src/` edits reach the GPU through GitHub. The loop
is: edit locally → commit + push → re-run the `git pull` cell below → re-run
the stage cell.

In [1]:
!nvidia-smi

Fri Sep  4 22:28:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Clone the repo

Public repo, so no credentials are needed. Re-running this cell pulls the
latest commit rather than re-cloning.

(If you ever flip it back to private, add a GitHub PAT with `repo` scope as a
Colab secret named `GITHUB_TOKEN` — the cell picks it up automatically.)

In [9]:
OWNER = "rushilpatra"
REPO = "llm-finetune-serve"
BRANCH = "main"

import os, subprocess

try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    TOKEN = os.environ.get("GITHUB_TOKEN")

auth = f"{TOKEN}@" if TOKEN else ""
url = f"https://{auth}github.com/{OWNER}/{REPO}.git"

if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--branch", BRANCH, url, REPO], check=True)
%cd /content/llm-finetune-serve
!git pull --ff-only

/content/llm-finetune-serve
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 1), reused 4 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 8.05 KiB | 4.02 MiB/s, done.
From https://github.com/rushilpatra/llm-finetune-serve
   b428224..f231b46  main       -> origin/main
Updating b428224..f231b46
Fast-forward
 notebooks/run.ipynb | 398 ++++++++++++++++++++++++++++++++++++++++++++++++----
 1 file changed, 374 insertions(+), 24 deletions(-)


## 2. Install

Colab preinstalls `torch` / `torchvision` / `torchaudio` built against one CUDA
version, and vLLM pulls a torch built against another. Mixing them raises

    RuntimeError: Detected that PyTorch and TorchAudio were compiled with
    different CUDA versions

on `import vllm`. So we uninstall all four — vLLM included, otherwise pip sees
it already installed, skips it, and never reinstalls the torch we just removed
— and let a clean vLLM install pull a matched set.

**This replaces torch, so the kernel must be restarted afterwards.** In the
browser Colab prompts you; **in VS Code it does not** — click the **↺ Restart**
button in the notebook toolbar yourself. Then re-run the clone cell above and
skip straight to the version check; the install is cached.

In [10]:
# vLLM owns the torch stack. vLLM is uninstalled too, so pip actually
# re-resolves it instead of treating the requirement as already satisfied.
!pip uninstall -q -y vllm torch torchvision torchaudio
!pip install -q vllm
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 16.6 MB/s eta 0:00:00:00:01


In [4]:
import torch, transformers

print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("cuda        ", torch.cuda.is_available())
if torch.cuda.is_available():
    # T4 (Turing) has no bf16 support; scripts detect this at runtime.
    print("gpu         ", torch.cuda.get_device_name(0))
    print("bf16        ", torch.cuda.is_bf16_supported())

# Import vLLM here rather than discovering a broken install 20 minutes into a run.
import vllm
print("vllm        ", vllm.__version__)

torch        2.13.0+cu130
transformers 5.16.1
cuda         True
gpu          NVIDIA A100-SXM4-40GB
bf16         True


## 3. Data smoke test

Prints split sizes, the 8-shot prefix length, and a sample prompt with the
round-trip answer-extraction check.

In [5]:
!python -m src.data --split val --limit 2

README.md: 100% 7.93k/7.93k [00:00<00:00, 17.3MB/s]

main/train-00000-of-00001.parquet: downloading bytes:   1% 32.7k/2.31M [00:01<01:31, 24.8kB/s]
main/train-00000-of-00001.parquet: downloading bytes: 100% 2.30M/2.30M [00:01<00:00, 1.65MB/s,  224kB/s  ]
main/train-00000-of-00001.parquet: reconstructing file: 100% 2.31M/2.31M [00:01<00:00, 1.65MB/s,  224kB/s  ]

main/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/419k [00:00<?, ?B/s]s]
main/test-00000-of-00001.parquet: downloading bytes: 100% 419k/419k [00:01<00:00, 325kB/s, 40.8kB/s  ]
main/test-00000-of-00001.parquet: reconstructing file: 100% 419k/419k [00:01<00:00, 325kB/s, 40.8kB/s  ]
Generating train split: 100% 7473/7473 [00:00<00:00, 379284.05 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 294898.03 examples/s]
split sizes:
  fewshot  8
  val      750
  train    6715
  test     1319

8-shot prefix: 3600 chars

re male. If there are 18 contestants in total, how many of them are male?
Answer: There are

## 4. Baseline: 8-shot prompting of the base model

Plumbing check first (no GPU, seconds), then the real run. Predictions stream
to `results/baseline_8shot.jsonl` as they are produced, so if the session dies
you re-run the same command and it picks up where it stopped.

In [6]:
!python -m src.evaluate --config configs/eval_baseline_8shot.yaml --dry-run --limit 4

!! DRY RUN: generations are gold completions, metrics are meaningless

4 examples, 0 already done, 4 to generate
  4/4

run               baseline_8shot  (val, n=4)
exact match       1.0000 +/- 0.0000
format adherence  1.0000 +/- 0.0000
wall clock        0.0s
gpu               NVIDIA A100-SXM4-40GB  peak 0.45 GB

predictions  results/baseline_8shot-dryrun.jsonl
metrics      results/baseline_8shot-dryrun.metrics.json


In [7]:
!python -m src.evaluate --config configs/eval_baseline_8shot.yaml

750 examples, 0 already done, 750 to generate
Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/llm-finetune-serve/src/evaluate.py", line 264, in <module>
    _main()
    ~~~~~^^
  File "/content/llm-finetune-serve/src/evaluate.py", line 252, in _main
    metrics = run(config, args.limit, out, args.dry_run)
  File "/content/llm-finetune-serve/src/evaluate.py", line 198, in run
    generations = generate(prompts, config)
  File "/content/llm-finetune-serve/src/evaluate.py", line 134, in generate
    from vllm import LLM, SamplingParams
  File "<frozen importlib._bootstrap>", line 1420, in _handle_fromlist
  File "/usr/local/lib/python3.13/dist-packages/vllm/__init__.py", line 70, in __getattr__
    module = import_module(module_name, __package__)
  File "/usr/lib/python3.13/importlib/__init__.py", line 88, in import_module
    return _bootstrap._gcd_import(name[level:], package, le

## 5. Save results off the VM

Colab VMs are ephemeral. Download `results/` and commit it from your laptop —
the per-example JSONL is what the paired bootstrap reads later.

In [8]:
!cd /content/llm-finetune-serve && zip -qr /content/results.zip results

from google.colab import files
files.download("/content/results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Next stages

Cells for training, merging, and benchmarking get added here as those scripts
land. Each is a single `!python -m src.<script> --config configs/<run>.yaml`.